In [1]:
!pip install -q transformers datasets huggingface_hub selfcheckgpt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets   = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

print(f"Token starts with: {secret_value_0[:5]}")
print(f"Token length: {len(secret_value_0)}")

login(token=secret_value_0)
print("Logged in to HuggingFace")

Token starts with: hf_Xa
Token length: 37
✓ Logged in to HuggingFace


In [3]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

CUDA available: True
Device: Tesla T4


In [4]:
from datasets import load_dataset

cnndm_splits = load_dataset('factshield-team/cache', 'cnndm_splits')
xsum_splits  = load_dataset('factshield-team/cache', 'xsum_splits')
fb_splits    = load_dataset('factshield-team/cache', 'faithbench_splits')

print(f"✓ CNN/DM train:      {len(cnndm_splits['train'])} docs")
print(f"✓ XSUM train:        {len(xsum_splits['train'])} docs")
print(f"✓ FaithBench train:  {len(fb_splits['train'])} docs")

README.md: 0.00B [00:00, ?B/s]

cnndm_splits/train-00000-of-00001.parque(…):   0%|          | 0.00/3.70M [00:00<?, ?B/s]

cnndm_splits/val-00000-of-00001.parquet:   0%|          | 0.00/760k [00:00<?, ?B/s]

cnndm_splits/test-00000-of-00001.parquet:   0%|          | 0.00/770k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1400 [00:00<?, ? examples/s]

...

faithbench_splits/test-00000-of-00001.pa(…):   0%|          | 0.00/109k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/120 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/120 [00:00<?, ? examples/s]

✓ CNN/DM train:      1400 docs
✓ XSUM train:        1400 docs
✓ FaithBench train:  560 docs


In [ ]:
import gc
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODELS = {
    'bart':    'facebook/bart-base',
    't5':      't5-small',
    'pegasus': 'google/pegasus-large',
}

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
HUB_REPO = 'factshield-team/cache'
CHECKPOINT_EVERY = 500
BATCH_SIZE = 16
K = 10
print(f"Using device: {DEVICE}")

def _keys(dataset):
    cols = dataset.column_names
    article_key = 'article' if 'article' in cols else ('document' if 'document' in cols else 'source')
    id_key      = 'id'      if 'id'      in cols else ('doc_id' if 'doc_id' in cols else None)
    return article_key, id_key

def load_model(name):
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODELS[name], dtype=torch.float16
    ).eval().to(DEVICE)
    model = torch.compile(model)
    tokenizer = AutoTokenizer.from_pretrained(MODELS[name])
    return model, tokenizer


def free_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


def generate_summaries(model, tokenizer, dataset, model_name, dataset_name,
                       batch_size=BATCH_SIZE):
    results = []
    article_key, id_key = _keys(dataset)

    for i in range(0, len(dataset), batch_size):
        batch    = dataset[i : i + batch_size]
        articles = batch[article_key]
        ids = [str(x) for x in batch[id_key]] if id_key else [str(i + j) for j in range(len(articles))]
        inputs = tokenizer(
            articles, return_tensors='pt', padding=True,
            truncation=True, max_length=512
        ).to(DEVICE)

        with torch.no_grad():
            out = model.generate(
                **inputs, num_beams=4,
                max_new_tokens=128, no_repeat_ngram_size=3
            )

        summaries = tokenizer.batch_decode(out, skip_special_tokens=True)
        for doc_id, article, summary in zip(ids, articles, summaries):
            results.append({'doc_id': doc_id, 'source_text': article, 'summary': summary})

        if i > 0 and i % CHECKPOINT_EVERY == 0:
            Dataset.from_list(results).push_to_hub(
                HUB_REPO,
                config_name=f'{model_name}_{dataset_name}_summaries',
                private=True
            )
            print(f"  checkpoint @ doc {i}")

    Dataset.from_list(results).push_to_hub(
        HUB_REPO,
        config_name=f'{model_name}_{dataset_name}_summaries',
        private=True
    )
    print(f"✓ {model_name}_{dataset_name}_summaries done ({len(results)} docs)")

def generate_k_samples(model, tokenizer, dataset, model_name, dataset_name,
                       K=K, batch_size=BATCH_SIZE):
    results = []
    article_key, id_key = _keys(dataset)

    for i in range(0, len(dataset), batch_size):
        batch    = dataset[i : i + batch_size]
        articles = batch[article_key]
        ids = [str(x) for x in batch[id_key]] if id_key else [str(i + j) for j in range(len(articles))]

        inputs = tokenizer(
            articles, return_tensors='pt', padding=True,
            truncation=True, max_length=512
        ).to(DEVICE)

        batch_samples = [[] for _ in range(len(articles))] 
        for _ in range(K):
            with torch.no_grad():
                out = model.generate(
                    **inputs, do_sample=True,
                    temperature=1.0, max_new_tokens=128
                )
            decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
            for j, s in enumerate(decoded):
                batch_samples[j].append(s)

        for doc_id, samples in zip(ids, batch_samples):
            row = {'doc_id': doc_id}
            for k, s in enumerate(samples):
                row[f'summary_k{k+1}'] = s
            results.append(row)

        if i > 0 and i % CHECKPOINT_EVERY == 0:
            Dataset.from_list(results).push_to_hub(
                HUB_REPO,
                config_name=f'{model_name}_{dataset_name}_k_samples',
                private=True
            )
            print(f"  checkpoint @ doc {i}")

    Dataset.from_list(results).push_to_hub(
        HUB_REPO,
        config_name=f'{model_name}_{dataset_name}_k_samples',
        private=True
    )
    print(f"✓ {model_name}_{dataset_name}_k_samples done ({len(results)} docs)")

def generate_with_scores(model, tokenizer, dataset, model_name, dataset_name,
                         batch_size=BATCH_SIZE):
    results = []
    article_key, id_key = _keys(dataset)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    for i in range(0, len(dataset), batch_size):
        batch    = dataset[i : i + batch_size]
        articles = batch[article_key]
        ids = [str(x) for x in batch[id_key]] if id_key else [str(i + j) for j in range(len(articles))]

        inputs = tokenizer(
            articles, return_tensors='pt', padding=True,
            truncation=True, max_length=512
        ).to(DEVICE)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                output_scores=True,
                return_dict_in_generate=True,
                num_beams=1,
                max_new_tokens=128
            )

        scores    = torch.stack(out.scores, dim=1)
        log_probs = scores.log_softmax(-1)
        token_ids = out.sequences[:, 1:]

        for b, doc_id in enumerate(ids):
            tids    = token_ids[b]
            seq_len = (tids != pad_id).sum().item()
            tids    = tids[:seq_len]
            chosen_lp = log_probs[b, range(seq_len), tids]
            results.append({
                'doc_id':         doc_id,
                'token_logprobs': chosen_lp.cpu().float().numpy().tobytes(),
                'seq_len':        seq_len,
            })

        if i > 0 and i % CHECKPOINT_EVERY == 0:
            Dataset.from_list(results).push_to_hub(
                HUB_REPO,
                config_name=f'{model_name}_{dataset_name}_token_scores',
                private=True
            )
            print(f"  checkpoint @ doc {i}")

    Dataset.from_list(results).push_to_hub(
        HUB_REPO,
        config_name=f'{model_name}_{dataset_name}_token_scores',
        private=True
    )
    print(f"✓ {model_name}_{dataset_name}_token_scores done ({len(results)} docs)")

print("✓ All functions defined")

Using device: cuda
✓ All functions defined


In [17]:
model, tokenizer = load_model('t5')
print("T5 loaded")

for dataset_name, splits in [('cnndm', cnndm_splits), ('xsum', xsum_splits), ('faithbench', fb_splits)]:
    print(f"\n--- T5 {dataset_name} ---")
    generate_summaries(model, tokenizer, splits['train'], 't5', dataset_name)
    generate_k_samples(model, tokenizer, splits['train'], 't5', dataset_name, K=K)
    generate_with_scores(model, tokenizer, splits['train'], 't5', dataset_name)

print("\nAll T5 configs pushed to factshield-team/cache")
free_model(model)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5 loaded

--- T5 faithbench ---


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ t5_faithbench_token_scores done (560 docs)

All T5 configs pushed to factshield-team/cache


In [18]:
model, tokenizer = load_model('bart')
print("BART loaded")

for dataset_name, splits in [('cnndm', cnndm_splits), ('xsum', xsum_splits), ('faithbench', fb_splits)]:
    print(f"\n--- BART {dataset_name} ---")
    generate_summaries(model, tokenizer, splits['train'], 'bart', dataset_name)
    generate_k_samples(model, tokenizer, splits['train'], 'bart', dataset_name, K=K)
    generate_with_scores(model, tokenizer, splits['train'], 'bart', dataset_name)

print("\nAll BART configs pushed to factshield-team/cache")
free_model(model)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ bart_faithbench_token_scores done (560 docs)

All BART configs pushed to factshield-team/cache


In [8]:
BATCH_SIZE = 4
model, tokenizer = load_model('pegasus')
print("pegasus loaded")

for dataset_name, splits in [('cnndm', cnndm_splits),('xsum', xsum_splits), ('faithbench', fb_splits)]:
    print(f"\n--- PEGASUS × {dataset_name} ---")
    generate_summaries(model, tokenizer, splits['train'], 'pegasus', dataset_name, batch_size=4)
    torch.cuda.empty_cache()
    gc.collect()
    generate_k_samples(model, tokenizer, splits['train'], 'pegasus', dataset_name, K=5, batch_size=4)
    torch.cuda.empty_cache()
    gc.collect()
    generate_with_scores(model, tokenizer, splits['train'], 'pegasus', dataset_name, batch_size=4)
    torch.cuda.empty_cache()
    gc.collect()

print("\nAll PEGASUS configs pushed to factshield-team/cache")
free_model(model)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

Error during conversion: ReadTimeout('The read operation timed out')
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-large
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tra

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

pegasus loaded

--- PEGASUS × cnndm ---


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ pegasus_cnndm_token_scores done (1400 docs)

All PEGASUS configs pushed to factshield-team/cache


In [ ]:
configs_to_check = [
    't5_cnndm_summaries',        't5_cnndm_k_samples',        't5_cnndm_token_scores',
    't5_xsum_summaries',         't5_xsum_k_samples',         't5_xsum_token_scores',
    't5_faithbench_summaries',   't5_faithbench_k_samples',   't5_faithbench_token_scores',
    'bart_cnndm_summaries',      'bart_cnndm_k_samples',      'bart_cnndm_token_scores',
    'bart_xsum_summaries',       'bart_xsum_k_samples',       'bart_xsum_token_scores',
    'bart_faithbench_summaries', 'bart_faithbench_k_samples', 'bart_faithbench_token_scores',
]

for config in configs_to_check:
    ds = load_dataset(HUB_REPO, config)
    print(f"{config}: {len(ds['train'])} rows")